# AfriSenti Sentiment Analysis Notebook

Experimental notebook- four-system comparison (TF-IDF + Logistic Regression, mBERT, LAFT, MAFT), weighted F1 as the primary score, official test-split reporting, a code-mixing diagnostic, and LIME-based error analysis. The proposal explicitly frames weighted F1 as the main metric and requires results on the official test splits, with disaggregation for code-mixed vs monolingual subsets and LIME error analysis.


In [8]:
import os
import re
import gc
import time
import math
import json
import random
import warnings
import subprocess
import sys
from pathlib import Path
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
packages = [
    "pandas", "numpy", "torch", "datasets", "evaluate", "transformers",
    "scikit-learn", "matplotlib", "seaborn", "lime", "nltk", "transformers[torch]"
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)
import evaluate
import matplotlib.pyplot as plt
import seaborn as sns
from lime.lime_text import LimeTextExplainer
import nltk
from nltk.corpus import words
from IPython.display import display, HTML

warnings.filterwarnings("ignore")
nltk.download("words", quiet=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Traceback (most recent call last):
  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/mnt/c/Users/LLM/repo/African-Langs-For-Sentiment-Analysis-Using-AfriSenti-Datasets-Sentiment-Analysis-In-African-Langs-/envv/lib/python3.12/site-packages/pip/__main__.py", line 22, in <module>
    from pip._internal.cli.main import main as _main
  File "/mnt/c/Users/LLM/repo/African-Langs-For-Sentiment-Analysis-Using-AfriSenti-Datasets-Sentiment-Analysis-In-African-Langs-/envv/lib/python3.12/site-packages/pip/_internal/cli/main.py", line 10, in <module>
    from pip._internal.cli.autocompletion import autocomplete
  File "/mnt/c/Users/LLM/repo/African-Langs-For-Sentiment-Analysis-Using-AfriSenti-Datasets-Sentiment-Analysis-In-African-Langs-/envv/lib/python3.12/site-packages/pip/_internal/cli/autocompletion.py", line 10, in <module>
    from pip._internal.cli.main_parser import create_main_parser
  File "/mnt/c/Users/LLM/repo/African-Langs-

KeyboardInterrupt: 

In [ ]:
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

    print("GPU memory cleared")
    print(torch.cuda.memory_summary())

In [ ]:
GLOBAL_SEED = 42
SEEDS = [42, 123, 456]#multi-run support
MULTI_SEED = False #set True for the full mean std reporting and this is the switcher

def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(GLOBAL_SEED)

### 1:- Repository and split loading
---
The loader below is strict about the official train/validation/test structure but it also has fallback logic so the notebook still runs if the repository uses slightly different split names. The shared task and proposal both require reporting on official test splits.


In [ ]:
REPO_URL = "https://github.com/afrisenti-semeval/afrisent-semeval-2023.git"
LOCAL_DIR = "afrisent-semeval-2023"

if not Path(LOCAL_DIR).exists():
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL], check=True)
else:
    print(f"Repository already exists at {LOCAL_DIR}")

BASE_DIR = Path(LOCAL_DIR) / "data"
LANGUAGES = ["hau", "yor", "ibo", "pcm", "swa"]

LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2} #multi-class
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def normalize_label(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, np.integer)):
        return int(value)
    text = str(value).strip().lower()
    text = text.replace("label_", "").replace("sentiment_", "")
    if text in LABEL2ID:
        return LABEL2ID[text]
    if text in {"neg", "negative", "0"}:
        return 0
    if text in {"neu", "neutral", "1"}:
        return 1
    if text in {"pos", "positive", "2"}:
        return 2
    try:
        return int(text)
    except ValueError:
        return np.nan

def detect_columns(df: pd.DataFrame):
    cols = {c.lower(): c for c in df.columns}
    text_col = next((cols[c] for c in ["tweet", "text", "sentence", "content"] if c in cols), None)
    label_col = next((cols[c] for c in ["label", "sentiment", "gold_label"] if c in cols), None)
    if text_col is None or label_col is None:
        raise ValueError(f"Couldn't detect text/label columns from {list(df.columns)}")
    return text_col, label_col

def load_split(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep="\t")
    text_col, label_col = detect_columns(df)
    df = df[[text_col, label_col]].copy()
    df.columns = ["tweet", "label"]
    df["tweet"] = df["tweet"].astype(str).fillna("")
    df["label"] = df["label"].apply(normalize_label)
    df = df.dropna(subset=["tweet", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df.reset_index(drop=True)

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

def load_language_bundle(lang: str):
    lang_dir = BASE_DIR / lang
    train_path = first_existing([lang_dir / "train.tsv"])
    val_path = first_existing([
        lang_dir / "dev.tsv",
        lang_dir / "val.tsv",
        lang_dir / "validation.tsv"
    ])
    test_path = first_existing([
        lang_dir / "test.tsv",
        lang_dir / "test_gold.tsv"
    ])

    if train_path is None:
        raise FileNotFoundError(f"Missing train split for {lang}")

    train_df = load_split(train_path)

    if val_path is None:
        # fallback: hold out 10% of the training split, stratified
        from sklearn.model_selection import train_test_split
        train_df, val_df = train_test_split(
            train_df,
            test_size=0.1,
            random_state=GLOBAL_SEED,
            stratify=train_df["label"]
        )
        train_df = train_df.reset_index(drop=True)
        val_df = val_df.reset_index(drop=True)
    else:
        val_df = load_split(val_path)

    if test_path is None:
        raise FileNotFoundError(f"Missing test split for {lang}")

    test_df = load_split(test_path)

    return {
        "train": train_df,
        "validation": val_df,
        "test": test_df
    }

afrisenti_pandas = {}
afrisenti_hf = {}

for lang in LANGUAGES:
    try:
        splits = load_language_bundle(lang)
        afrisenti_pandas[lang] = splits
        afrisenti_hf[lang] = DatasetDict({
            split_name: Dataset.from_pandas(df, preserve_index=False)
            for split_name, df in splits.items()
        })
        print(f"[{lang.upper()}] train={len(splits['train'])}, val={len(splits['validation'])}, test={len(splits['test'])}")
    except Exception as e:
        print(f"[{lang.upper()}] skipped: {e}")

summary_rows = []
for lang, splits in afrisenti_pandas.items():
    for split_name, df in splits.items():
        summary_rows.append({
            "language": lang,
            "split": split_name,
            "n": len(df),
            "class_dist": df["label"].value_counts().sort_index().to_dict()
        })

summary_df = pd.DataFrame(summary_rows)
display(summary_df)

### 2:- Evaluation helpers functions

Weighted F1 is the primary metric along with macro F1, precision, recall, and accuracy reported alongside it. The shared-task evaluation emphasis.

In [ ]:

f1_weighted_metric = evaluate.load("f1")
f1_macro_metric = evaluate.load("f1")
accuracy_metric = evaluate.load("accuracy")

def compute_metrics_from_arrays(predictions: np.ndarray, labels: np.ndarray):
    preds = np.argmax(predictions, axis=-1)
    weighted = precision_recall_fscore_support(labels, preds, average="weighted", zero_division=0)
    macro = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    acc = accuracy_score(labels, preds)
    return {
        "accuracy": float(acc),
        "weighted_precision": float(weighted[0]),
        "weighted_recall": float(weighted[1]),
        "f1_weighted": float(weighted[2]),
        "macro_precision": float(macro[0]),
        "macro_recall": float(macro[1]),
        "macro_f1": float(macro[2]),
    }

def pretty_metrics(metrics: dict):
    return {k: round(v, 4) for k, v in metrics.items()}

def make_hf_dataset(df: pd.DataFrame):
    ds = Dataset.from_pandas(df[["tweet", "label"]].rename(columns={"label": "labels"}), preserve_index=False)
    return ds

def tokenize_dataset(dataset: Dataset, tokenizer, max_length: int = 128):
    return dataset.map(
    lambda batch: tokenizer(
        batch["tweet"],
        truncation=True,
        padding=False,
        max_length=max_length
    ),
    batched=True,
    load_from_cache_file=False,
    keep_in_memory=False
)

def format_tokenized_dataset(tokenized: Dataset):
    cols = [c for c in ["input_ids", "attention_mask", "labels"] if c in tokenized.column_names]
    return tokenized.remove_columns([c for c in tokenized.column_names if c not in cols]).with_format("torch")

def concat_language_splits(split_name: str):
    return concatenate_datasets([afrisenti_hf[lang][split_name] for lang in afrisenti_hf.keys()])

class WeightedTrainer(Trainer):
    def __init__(self, class_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


### 3:- Baseline system: TF-IDF + Logistic Regression

The proposal lists TF-IDF+Logistic Regression as the first baseline, with unigram+bigram features and class weighting

In [ ]:

def run_tfidf_lr(train_df, val_df, test_df, seed=42, max_features=50000):
    set_seed(seed)

    pipe = Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=max_features)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=seed))
    ])

    X_train = train_df["tweet"].fillna("").tolist()
    y_train = train_df["label"].tolist()
    X_val = val_df["tweet"].fillna("").tolist()
    y_val = val_df["label"].tolist()
    X_test = test_df["tweet"].fillna("").tolist()
    y_test = test_df["label"].tolist()

    pipe.fit(X_train, y_train)

    val_pred = pipe.predict(X_val)
    test_pred = pipe.predict(X_test)

    val_metrics = compute_metrics_from_arrays(np.eye(3)[val_pred], np.array(y_val))
    test_metrics = compute_metrics_from_arrays(np.eye(3)[test_pred], np.array(y_test))

    return {
        "model": pipe,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
        "test_predictions": test_pred
    }

baseline_rows = []
baseline_models = {}

for lang, splits in afrisenti_pandas.items():
    result = run_tfidf_lr(splits["train"], splits["validation"], splits["test"], seed=GLOBAL_SEED)
    baseline_models[lang] = result["model"]
    row = {"language": lang.upper(), **{f"test_{k}": v for k, v in result["test_metrics"].items()}}
    baseline_rows.append(row)

baseline_df = pd.DataFrame(baseline_rows)
display(baseline_df.sort_values("test_f1_weighted", ascending=False))


### 4:- Transformer systems: mBERT, LAFT, and MAFT

The transformer comparison has two language settings mBERT and LAFT plus one joint multilingual setting, MAFT. 

AfroXLMR is the Africa centric model & mBERT is the standard multilingual reference.


In [ ]:

AFROXLMR_MODEL = "Davlan/afro-xlmr-base"
MBERT_MODEL = "bert-base-multilingual-cased"

MAX_LENGTH = 64
TRAIN_BATCH_SIZE = 2
EVAL_BATCH_SIZE = 2
GRAD_ACCUMULATION = 4
LR = 2e-5
EPOCHS = 3
EARLY_STOPPING = 2

def build_class_weights(train_df: pd.DataFrame):
    y = train_df["label"].values
    classes = np.array(sorted(train_df["label"].unique()))
    weights = compute_class_weight(class_weight="balanced", classes=classes, y=y)
    return torch.tensor(weights, dtype=torch.float)

def train_transformer_model(
    model_name,
    train_df,
    val_df,
    test_df,
    output_dir,
    seed=42,
    max_length=MAX_LENGTH
):

    set_seed(seed)

    trainer = None
    model = None
    tokenizer = None
    train_ds = None
    val_ds = None
    test_ds = None
    test_output = None

    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)

        train_ds = format_tokenized_dataset(
            tokenize_dataset(make_hf_dataset(train_df), tokenizer, max_length)
        )

        val_ds = format_tokenized_dataset(
            tokenize_dataset(make_hf_dataset(val_df), tokenizer, max_length)
        )

        test_ds = format_tokenized_dataset(
            tokenize_dataset(make_hf_dataset(test_df), tokenizer, max_length)
        )

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=3,
            id2label=ID2LABEL,
            label2id=LABEL2ID
        )

        class_weights = build_class_weights(train_df)

        training_args = TrainingArguments(
            output_dir=output_dir,
            learning_rate=LR,
            per_device_train_batch_size=TRAIN_BATCH_SIZE,
            per_device_eval_batch_size=EVAL_BATCH_SIZE,
            gradient_accumulation_steps=GRAD_ACCUMULATION,
            num_train_epochs=EPOCHS,
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="no",
            logging_steps=50,
            report_to="none",
            fp16=torch.cuda.is_available(),
            seed=seed
        )

        trainer = WeightedTrainer(
            model=model,
            args=training_args,
            train_dataset=train_ds,
            eval_dataset=val_ds,
            data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
            compute_metrics=lambda pred: compute_metrics_from_arrays(
                pred.predictions,
                pred.label_ids
            ),
            processing_class=tokenizer,
            class_weights=class_weights
        )

        start = time.time()

        trainer.train()

        val_metrics = trainer.evaluate(val_ds)

        test_output = trainer.predict(test_ds)

        test_metrics = compute_metrics_from_arrays(
            test_output.predictions,
            test_output.label_ids
        )

        results = {
            "val_metrics": val_metrics,
            "test_metrics": test_metrics,
            "test_predictions": np.argmax(
                test_output.predictions,
                axis=-1
            ).tolist(),
            "test_labels": test_output.label_ids.tolist(),
            "elapsed_sec": time.time() - start
        }

        return results

    finally:
        del trainer
        del model
        del tokenizer
        del train_ds
        del val_ds
        del test_ds
        del test_output

        torch.cuda.empty_cache()
        gc.collect()

def evaluate_one_seed(seed=42):
    results = {
        "mbert": {},
        "laft": {},
        "maft": {}
    }

    # mBERT and LAFT are trained per language
    for lang, splits in afrisenti_pandas.items():
        print(f"\n:-:-:-: Training mBERT for {lang.upper()} :-:-:-:")
        mbert_out = train_transformer_model(
            MBERT_MODEL,
            splits["train"],
            splits["validation"],
            splits["test"],
            output_dir=f"./outputs/mbert_{lang}_seed{seed}",
            seed=seed
        )
        results["mbert"][lang] = mbert_out
        results_to_save = {
            "val_metrics": mbert_out["val_metrics"],
            "test_metrics": mbert_out["test_metrics"],
            "elapsed_sec": mbert_out["elapsed_sec"],
            "test_predictions": mbert_out["test_predictions"].tolist(),
            "test_labels": mbert_out["test_labels"].tolist()
        }

        with open(f"mbert_{lang}_results.json", "w") as f:
            json.dump(results_to_save, f, indent=2)
        try:
            del trainer
        except:
            pass
        try:
            del model
        except:
            pass
        torch.cuda.empty_cache()
        gc.collect()

        print(f"\n:-:-:-: Training LAFT (AfroXLMR) for {lang.upper()} :-:-:-:")
        laft_out = train_transformer_model(
            AFROXLMR_MODEL,
            splits["train"],
            splits["validation"],
            splits["test"],
            output_dir=f"./outputs/laft_{lang}_seed{seed}",
            seed=seed
        )
        results["laft"][lang] = laft_out
        results_to_save = {
            "val_metrics": mbert_out["val_metrics"],
            "test_metrics": mbert_out["test_metrics"],
            "elapsed_sec": mbert_out["elapsed_sec"],
            "test_predictions": mbert_out["test_predictions"].tolist(),
            "test_labels": mbert_out["test_labels"].tolist()
        }

        with open(f"mbert_{lang}_results.json", "w") as f:
            json.dump(results_to_save, f, indent=2)

# 3. If you need to save the actual model for later, do this instead:
# mbert_out["trainer"].save_model(f"./outputs/mbert_{lang}_saved_model")

        try:
            del model
        except:
            pass

        torch.cuda.empty_cache()
        gc.collect()
    # MAFT: joint training on all languages
    print("\n:-:-:-: Training MAFT (joint AfroXLMR) :-:-:-:")
    tokenizer = AutoTokenizer.from_pretrained(AFROXLMR_MODEL)

    maft_train = concat_language_splits("train")
    maft_val = concat_language_splits("validation", MAX_LENGTH)
    maft_test_by_lang = {lang: afrisenti_hf[lang]["test"] for lang in afrisenti_hf.keys()}
    MAFT_MAX_LENGTH = 48
    maft_train_ds = format_tokenized_dataset(tokenize_dataset(maft_train, tokenizer, MAFT_MAX_LENGTH))
    maft_val_ds = format_tokenized_dataset(tokenize_dataset(maft_val, tokenizer, MAFT_MAX_LENGTH))

    model = AutoModelForSequenceClassification.from_pretrained(
        AFROXLMR_MODEL,
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID
    )
    model.gradient_checkpointing_enable()

    class_weights = build_class_weights(pd.concat([afrisenti_pandas[lang]["train"] for lang in afrisenti_pandas.keys()], ignore_index=True))

    training_args = TrainingArguments(
        output_dir=f"./outputs/maft_seed{seed}",
        learning_rate=LR,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="no",
        logging_steps=50,
        load_best_model_at_end=False,
        metric_for_best_model="f1_weighted",
        greater_is_better=True,
        report_to="none",
        fp16=torch.cuda.is_available(),
        save_total_limit=2,
        seed=seed,
        eval_accumulation_steps=4,
        torch_compile=False,
    )

    maft_trainer = WeightedTrainer(
        model=model,
        args=training_args,
        train_dataset=maft_train_ds,
        eval_dataset=maft_val_ds,
        data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
        compute_metrics=lambda pred: compute_metrics_from_arrays(pred.predictions, pred.label_ids),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING)],
        processing_class=tokenizer,
        class_weights=class_weights
    )

    start = time.time()
    trainer.train(resume_from_checkpoint=True)
    maft_results = {}
    for lang, test_ds in maft_test_by_lang.items():
        tok_test = format_tokenized_dataset(tokenize_dataset(test_ds, tokenizer, MAFT_MAX_LENGTH))
        pred = maft_trainer.predict(tok_test)
        maft_results[lang] = {
            "test_metrics": compute_metrics_from_arrays(pred.predictions, pred.label_ids),
            "test_predictions": np.argmax(pred.predictions, axis=-1),
            "test_labels": pred.label_ids
        }

    results["maft"]["trainer"] = maft_trainer
    results["maft"]["tokenizer"] = tokenizer
    results["maft"]["by_language"] = maft_results
    results["maft"]["elapsed_sec"] = time.time() - start
    return results

# Run once by default set MULTI_SEED=True to repeat with three seeds and average
if MULTI_SEED:
    all_seed_runs = []
    for seed in SEEDS:
        print(f"\n================ SEED {seed} ================\n")
        all_seed_runs.append(evaluate_one_seed(seed=seed))
    print("Completed multi-seed training.")
else:
    single_run = evaluate_one_seed(seed=GLOBAL_SEED)
    print("Single-seed training completed.")


### 5:- Results tables and benchmark reporting

Mean +- standard deviation over three runs, per-language reporting, and comparison against the official test splits

The helper cells below make that possible without changing the modelling code.

In [ ]:

def build_result_table_baseline():
    return baseline_df.copy()

def build_result_table_transformer(run_obj, system_name):
    rows = []
    if system_name in {"mbert", "laft"}:
        for lang, out in run_obj[system_name].items():
            rows.append({
                "language": lang.upper(),
                "system": system_name.upper(),
                **{f"test_{k}": v for k, v in out["test_metrics"].items()}
            })
    elif system_name == "maft":
        for lang, out in run_obj["maft"]["by_language"].items():
            rows.append({
                "language": lang.upper(),
                "system": "MAFT",
                **{f"test_{k}": v for k, v in out["test_metrics"].items()}
            })
    return pd.DataFrame(rows)

mbert_results_df = build_result_table_transformer(single_run, "mbert")
laft_results_df = build_result_table_transformer(single_run, "laft")
maft_results_df = build_result_table_transformer(single_run, "maft")

display(mbert_results_df.sort_values("test_f1_weighted", ascending=False))
display(laft_results_df.sort_values("test_f1_weighted", ascending=False))
display(maft_results_df.sort_values("test_f1_weighted", ascending=False))


### 6:- Code-mixing diagnostic
--- 
The proposal defines a code-mixing diagnostic based on the share of English tokens, and expects a disaggregated analysis of monolingual versus code-mixed subsets. The notebook keeps that logic in a reusable diagnostic layer.


In [ ]:

english_vocab = set(w.lower() for w in words.words() if len(w) > 1 or w.lower() in {"a", "i"})

def cmi_features(tweet):
    if not isinstance(tweet, str):
        return pd.Series({"total_tokens": 0, "english_tokens": 0, "english_fraction": 0.0, "cmi": 0.0})

    text = re.sub(r"http\S+", " ", tweet)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#\w+", " ", text)
    text = re.sub(r"\d+", " ", text)

    tokens = re.findall(r"\b[a-zA-Z]+\b", text.lower())
    total = len(tokens)
    if total == 0:
        return pd.Series({"total_tokens": 0, "english_tokens": 0, "english_fraction": 0.0, "cmi": 0.0})

    eng = sum(1 for t in tokens if t in english_vocab)
    frac = eng / total
    afr = total - eng
    cmi = 100 * (1 - max(eng, afr) / total)

    return pd.Series({
        "total_tokens": total,
        "english_tokens": eng,
        "english_fraction": frac,
        "cmi": round(cmi, 2)
    })

def annotate_code_mixing(df: pd.DataFrame, threshold=0.20):
    out = df.copy()
    out = pd.concat([out, out["tweet"].apply(cmi_features)], axis=1)
    out["is_code_mixed"] = out["english_fraction"] > threshold
    out["mixing_bucket"] = pd.cut(
        out["cmi"],
        bins=[-1, 0.1, 15, 30, 100],
        labels=["Pure", "Low", "Medium", "High"]
    )
    return out

# standard diagnostic target: Nigerian Pidgin from code-mixing discussion with a 20% threshold
diag_lang = "pcm" if "pcm" in afrisenti_pandas else list(afrisenti_pandas.keys())[0]
diag_test = annotate_code_mixing(afrisenti_pandas[diag_lang]["test"], threshold=0.20)

display(diag_test[["tweet", "label", "english_fraction", "cmi", "is_code_mixed", "mixing_bucket"]].head())
print("Code-mixed:", int(diag_test["is_code_mixed"].sum()))
print("Monolingual:", int((~diag_test["is_code_mixed"]).sum()))


In [ ]:

def predict_dataframe_with_transformer(trainer, tokenizer, df: pd.DataFrame, max_length=128):
    ds = format_tokenized_dataset(tokenize_dataset(make_hf_dataset(df), tokenizer, max_length))
    pred = trainer.predict(ds)
    metrics = compute_metrics_from_arrays(pred.predictions, pred.label_ids)
    predicted_labels = np.argmax(pred.predictions, axis=-1)
    return metrics, predicted_labels

def plot_code_mixing_gap(model_bundle, df_with_mixing: pd.DataFrame, title: str):
    pure_df = df_with_mixing[~df_with_mixing["is_code_mixed"]].copy()
    mixed_df = df_with_mixing[df_with_mixing["is_code_mixed"]].copy()

    if len(pure_df) == 0 or len(mixed_df) == 0:
        print("Insufficient samples in one of the subsets.")
        return None

    metrics_pure, _ = predict_dataframe_with_transformer(
        model_bundle["trainer"], model_bundle["tokenizer"], pure_df
    )
    metrics_mixed, _ = predict_dataframe_with_transformer(
        model_bundle["trainer"], model_bundle["tokenizer"], mixed_df
    )

    gap = metrics_pure["f1_weighted"] - metrics_mixed["f1_weighted"]
    print(title)
    print("Pure weighted F1 :", round(metrics_pure["f1_weighted"], 4))
    print("Mixed weighted F1:", round(metrics_mixed["f1_weighted"], 4))
    print("Delta F1         :", round(gap, 4))
    return {
        "pure": metrics_pure,
        "mixed": metrics_mixed,
        "delta_f1": gap
    }

# Code-mixing analysis on the diagnostic language for the two AfroXLMR variants
if "maft" in single_run:
    maft_bundle = {
        "trainer": single_run["maft"]["trainer"],
        "tokenizer": single_run["maft"]["tokenizer"]
    }
    laft_bundle = {
        "trainer": single_run["laft"][diag_lang]["trainer"],
        "tokenizer": single_run["laft"][diag_lang]["tokenizer"]
    }

    maft_gap = plot_code_mixing_gap(maft_bundle, diag_test, f"MAFT on {diag_lang.upper()}")
    laft_gap = plot_code_mixing_gap(laft_bundle, diag_test, f"LAFT on {diag_lang.upper()}")


### 7:- LIME explanation and misclassification analysis

Tokens driving misclassification and with error taxonomy analysis via LIME


In [ ]:

def make_lime_predict_fn(model, tokenizer, max_length=64, batch_size=8):
    model = model.to(device)
    model.eval()

    def predict(texts):
        all_probs = []
        for start in range(0, len(texts), batch_size):
            chunk = texts[start:start + batch_size]
            enc = tokenizer(
                chunk,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_length
            )
            enc = {k: v.to(device) for k, v in enc.items()}
            with torch.no_grad():
                logits = model(**enc).logits
                probs = torch.softmax(logits, dim=-1).cpu().numpy()
            all_probs.append(probs)
        return np.vstack(all_probs)

    return predict

def find_misclassified_example(trainer, tokenizer, df: pd.DataFrame):
    probs, preds = None, None
    metrics, preds = predict_dataframe_with_transformer(trainer, tokenizer, df)
    true_labels = df["label"].to_numpy()
    mis_idx = np.where(preds != true_labels)[0]
    if len(mis_idx) == 0:
        return None
    idx = int(np.random.choice(mis_idx))
    return {
        "index": idx,
        "text": df.iloc[idx]["tweet"],
        "true_label": int(true_labels[idx]),
        "pred_label": int(preds[idx]),
        "metrics": metrics
    }

def explain_with_lime(trainer, tokenizer, text, class_names=("negative", "neutral", "positive"), num_features=10):
    predict_fn = make_lime_predict_fn(trainer.model, tokenizer)
    explainer = LimeTextExplainer(class_names=list(class_names))
    explanation = explainer.explain_instance(
        text_instance=text,
        classifier_fn=predict_fn,
        num_features=num_features,
        top_labels=3
    )
    return explanation

# Prefer the diagnostic language for LIME
lime_bundle = single_run["maft"]["trainer"], single_run["maft"]["tokenizer"]
lime_trainer, lime_tokenizer = lime_bundle

misclassified = find_misclassified_example(lime_trainer, lime_tokenizer, afrisenti_pandas[diag_lang]["test"])
if misclassified is None:
    print("No misclassification found in this sample.")
else:
    print("True label     :", ID2LABEL[misclassified["true_label"]])
    print("Predicted label:", ID2LABEL[misclassified["pred_label"]])
    print("Text:\n", misclassified["text"])
    lime_exp = explain_with_lime(lime_trainer, lime_tokenizer, misclassified["text"])
    display(HTML(lime_exp.as_html()))


### 8:- Comparison against SemEval-2023

The proposal says the final results should be interpretable against the SemEval-2023 leaderboard, so the final cell below provides a lightweight comparison. 

The official evaluation script uses the weighted F1 / support-weighted score as the key score which is why the notebook centers that metric throughout.

In [ ]:

SEMEVAL_REFERENCE = {
    "best_monolingual_weighted_f1": 0.7131,
    "best_zero_shot_weighted_f1": 0.5815
}

def compare_to_reference(score, reference=SEMEVAL_REFERENCE["best_monolingual_weighted_f1"]):
    return round(score - reference, 4)

final_summary = []

for lang in baseline_df["language"]:
    b = baseline_df[baseline_df["language"] == lang].iloc[0]
    m = mbert_results_df[mbert_results_df["language"] == lang].iloc[0]
    l = laft_results_df[laft_results_df["language"] == lang].iloc[0]
    a = maft_results_df[maft_results_df["language"] == lang].iloc[0]
    final_summary.append({
        "language": lang,
        "TFIDF_LR": round(b["test_f1_weighted"], 4),
        "mBERT": round(m["test_f1_weighted"], 4),
        "LAFT": round(l["test_f1_weighted"], 4),
        "MAFT": round(a["test_f1_weighted"], 4),
        "MAFT_vs_SemEval_Mono": compare_to_reference(a["test_f1_weighted"])
    })

final_summary_df = pd.DataFrame(final_summary)
display(final_summary_df.sort_values("MAFT", ascending=False))
